# Tutorial 5: Running Inference with Multiprocessing

This tutorial demonstrates how to run inference using `mp_inference.run()`, the multiprocessing
drop-in for `inference.run()`.  We run serial and MP inference on the same event and bank,
then compare posteriors and timing.

## What changes vs serial (`inference.run_and_profile`)

| Stage | Serial | MP |
|-------|--------|----|
| 1 – Setup | serial | serial |
| 2 – Incoherent | serial batches | **Pool over bank chunks** |
| 3 – Cross-bank cut | serial | serial |
| 4 – Extrinsic | serial | serial (or parallel with `n_ext_workers>1`) |
| 5 – Coherent | serial i_blocks | **Pool, one worker per i_block (thin)** |
| 6 – Postprocess | serial | serial |

Expected speedup: ~`n_workers`× on the coherent stage, moderate gain on incoherent.

In [ ]:
# Cell 1 — env vars MUST come before any numpy/scipy import
import os
os.environ["OMP_NUM_THREADS"]      = "1"
os.environ["MKL_NUM_THREADS"]      = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"]  = "1"

In [ ]:
import sys
import time
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")

sys.path.append("../..")

from cogwheel import data, gw_utils, gw_plotting, utils, prior_ratio
from cogwheel.gw_prior import IntrinsicIASPrior
from dot_pe import inference, mp_inference, config, waveform_banks
from dot_pe.power_law_mass_prior import PowerLawIntrinsicIASPrior

ARTIFACTS_DIR = Path("./artifacts_mp")
ARTIFACTS_DIR.mkdir(exist_ok=True)
print("Imports OK")

## Step 1: Create event and bank

Same injection as notebook 03 — reuses existing artifacts if already present.

In [ ]:
eventname = "tutorial_mp_event"
chirp_mass = 20.0
q = 0.7
m1, m2 = gw_utils.mchirpeta_to_m1m2(chirp_mass, gw_utils.q_to_eta(q))

injection_par_dic = dict(
    m1=m1,
    m2=m2,
    ra=0.5,
    dec=0.5,
    iota=np.pi / 3,
    psi=1.0,
    phi_ref=12.0,
    s1z=0.3,
    s2z=0.3,
    s1x_n=0.1,
    s1y_n=0.2,
    s2x_n=0.3,
    s2y_n=-0.2,
    l1=0.0,
    l2=0.0,
    tgps=0.0,
    f_ref=50.0,
    d_luminosity=2000.0,
    t_geocenter=0.0,
)

event_path = ARTIFACTS_DIR / f"{eventname}.npz"
if not event_path.exists():
    event_data = data.EventData.gaussian_noise(
        eventname=eventname,
        detector_names="HLV",
        duration=120.0,
        asd_funcs=["asd_H_O3", "asd_L_O3", "asd_V_O3"],
        tgps=0.0,
        fmax=1600.0,
        seed=20260314,
    )
    event_data.inject_signal(injection_par_dic, "IMRPhenomXPHM")
    snr = np.sqrt(
        2 * (event_data.injection["d_h"] - 0.5 * event_data.injection["h_h"]).sum()
    )
    print(f"Injection SNR: {snr:.2f}")
    event_data.to_npz(filename=event_path, overwrite=True)
    print(f"Saved event: {event_path}")
else:
    print(f"Reusing existing event: {event_path}")

In [ ]:
bank_dir   = ARTIFACTS_DIR / "bank"
bank_size  = 2**16
mchirp_min, mchirp_max = 15, 30
q_min      = 0.2
f_ref      = 50.0
seed_bank  = 777
approximant = "IMRPhenomXPHM"

if not (bank_dir / "intrinsic_sample_bank.feather").exists():
    bank_dir.mkdir(parents=True, exist_ok=True)

    powerlaw_prior = PowerLawIntrinsicIASPrior(
        mchirp_range=(mchirp_min, mchirp_max), q_min=q_min, f_ref=f_ref,
    )
    ias_prior = IntrinsicIASPrior(
        mchirp_range=(mchirp_min, mchirp_max), q_min=q_min, f_ref=f_ref,
    )
    pr_ratio = prior_ratio.PriorRatio(ias_prior, powerlaw_prior)
    prior_ratio._remove_matching_items(
        pr_ratio._numerator_subpriors, pr_ratio._denominator_subpriors
    )

    print(f"Generating {bank_size:,} bank samples...")
    bank_samples = powerlaw_prior.generate_random_samples(bank_size, seed=seed_bank, return_lnz=False)
    bank_samples["mchirp"]  = gw_utils.m1m2_to_mchirp(bank_samples["m1"], bank_samples["m2"])
    bank_samples["lnq"]     = np.log(bank_samples["m2"] / bank_samples["m1"])
    bank_samples["chieff"]  = gw_utils.chieff(*bank_samples[["m1", "m2", "s1z", "s2z"]].values.T)
    bank_samples["log_prior_weights"] = bank_samples.apply(
        lambda row: pr_ratio.ln_prior_ratio(**row.to_dict()), axis=1
    )

    bank_columns = ["m1", "m2", "s1z", "s1x_n", "s1y_n", "s2z", "s2x_n", "s2y_n", "iota", "log_prior_weights"]
    samples_path = bank_dir / "intrinsic_sample_bank.feather"
    bank_samples[bank_columns].to_feather(samples_path)

    bank_config = {
        "bank_size": bank_size,
        "mchirp_min": mchirp_min, "mchirp_max": mchirp_max,
        "q_min": q_min, "f_ref": f_ref,
        "fbin": config.DEFAULT_FBIN.tolist(),
        "approximant": approximant,
        "m_arr": [2, 1, 3, 4],
        "seed": seed_bank,
    }
    with open(bank_dir / "bank_config.json", "w") as f:
        json.dump(bank_config, f, indent=4)

    print(f"Generating waveforms (4 cores)...")
    waveform_banks.create_waveform_bank_from_samples(
        samples_path=samples_path,
        bank_config_path=bank_dir / "bank_config.json",
        waveform_dir=bank_dir / "waveforms",
        n_pool=4, blocksize=4096, approximant=approximant,
    )
    print("Bank ready.")
else:
    print(f"Reusing existing bank: {bank_dir}")

## Step 2: Serial baseline

In [ ]:
N_EXT = 1024
N_PHI = 50
N_T = 128
SEED = 42
MCHIRP = chirp_mass
BS = 2048

serial_rundir_root = ARTIFACTS_DIR / "run_serial"
serial_rundir_root.mkdir(exist_ok=True)

t_serial_start = time.time()
serial_rundir = inference.run_and_profile(
    event=event_path,
    bank_folder=bank_dir,
    n_ext=N_EXT,
    n_phi=N_PHI,
    n_t=N_T,
    blocksize=BS,
    single_detector_blocksize=BS,
    seed=SEED,
    event_dir=str(serial_rundir_root),
    mchirp_guess=MCHIRP,
    max_incoherent_lnlike_drop=20,
    max_bestfit_lnlike_diff=20,
    draw_subset=True,
)
t_serial = time.time() - t_serial_start

serial_summary = utils.read_json(serial_rundir / "summary_results.json")
print(f"\nSerial wall time : {t_serial:.1f} s")
print(f"ln_evidence      : {serial_summary['ln_evidence']:.4f}")
print(f"n_effective      : {serial_summary['n_effective']:.1f}")

## Step 3: MP run with `mp_inference.run()`

In [ ]:
N_WORKERS = 4

mp_rundir_root = ARTIFACTS_DIR / "run_mp"
mp_rundir_root.mkdir(exist_ok=True)

t_mp_start = time.time()
mp_rundir = mp_inference.run(
    event=event_path,
    bank_folder=bank_dir,
    n_ext=N_EXT,
    n_phi=N_PHI,
    n_t=N_T,
    blocksize=BS,
    single_detector_blocksize=BS,
    n_workers=N_WORKERS,
    n_ext_workers=1,
    seed=SEED,
    event_dir=str(mp_rundir_root),
    mchirp_guess=MCHIRP,
    max_incoherent_lnlike_drop=20,
    max_bestfit_lnlike_diff=20,
    draw_subset=True,
)
t_mp = time.time() - t_mp_start

mp_summary = utils.read_json(mp_rundir / "summary_results.json")
print(f"\nMP wall time ({N_WORKERS} workers) : {t_mp:.1f} s")
print(f"ln_evidence                    : {mp_summary['ln_evidence']:.4f}")
print(f"n_effective                    : {mp_summary['n_effective']:.1f}")

## Step 4: Comparison table

In [ ]:
rows = [
    {"mode": "serial",          "n_workers": 1,         "wall_s": t_serial,
     "ln_evidence": serial_summary["ln_evidence"],
     "n_effective": serial_summary["n_effective"],
     "n_effective_i": serial_summary["n_effective_i"],
     "n_effective_e": serial_summary["n_effective_e"]},
    {"mode": f"mp",              "n_workers": N_WORKERS, "wall_s": t_mp,
     "ln_evidence": mp_summary["ln_evidence"],
     "n_effective": mp_summary["n_effective"],
     "n_effective_i": mp_summary["n_effective_i"],
     "n_effective_e": mp_summary["n_effective_e"]},
]
df_cmp = pd.DataFrame(rows).set_index("mode")
df_cmp["speedup"] = t_serial / df_cmp["wall_s"]
df_cmp["ln_evidence_diff"] = df_cmp["ln_evidence"] - serial_summary["ln_evidence"]

display(df_cmp.round(3))

## Step 5: Corner plot — serial vs MP

In [ ]:
true_mchirp = gw_utils.m1m2_to_mchirp(injection_par_dic["m1"], injection_par_dic["m2"])
true_lnq    = np.log(injection_par_dic["m2"] / injection_par_dic["m1"])
true_chieff = gw_utils.chieff(
    injection_par_dic["m1"], injection_par_dic["m2"],
    injection_par_dic["s1z"], injection_par_dic["s2z"],
)
true_values = injection_par_dic | {"mchirp": true_mchirp, "lnq": true_lnq, "chieff": true_chieff}

serial_samples = pd.read_feather(serial_rundir / "samples.feather")
mp_samples     = pd.read_feather(mp_rundir     / "samples.feather")

params = ["mchirp", "lnq", "chieff", "iota", "ra", "dec", "d_luminosity"]

corner_plot = gw_plotting.MultiCornerPlot(
    [serial_samples, mp_samples],
    params=params,
    smooth=1.0,
    labels=["Serial", f"MP (n={N_WORKERS})"],
)
corner_plot.plot(max_figsize=7)
corner_plot.scatter_points(true_values, colors="red", marker=".", s=200, label="Injection")
plt.suptitle("Serial vs MP posteriors", y=1.01)
plt.show()